# 04 — Grafo e hipóteses

Carrega no Neo4j a parte avaliada do grafo (respostas das políticas, notas e fatos extraídos na Fase 4) e responde às hipóteses do projeto com consultas Cypher.

A lógica está em `src/preparar_grafo.py`, `src/grafo.py`, `src/consultas.py` e `src/validar_grafo.py`; as consultas ficam em `cypher/consultas/`, uma por arquivo. Este notebook executa as etapas na ordem:

1. carga do grafo;
2. validação do grafo contra os CSVs;
3. H1 — dado neural como categoria específica;
4. H2 — terceiros e países dos dados;
5. H3 — garantias frente às leis de referência;
6. pergunta complementar — os dois grupos;
7. para onde vai o sinal neural.

**Atenção:** as respostas vêm da avaliação por modelo de linguagem, ainda não validada contra a avaliação manual (issue #16). Os números abaixo são a leitura do modelo, não um resultado validado.

In [1]:
from pathlib import Path
import os
import sys

# Os caminhos do projeto são relativos à raiz do repositório.
RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

import pandas as pd

from src.conexao_neo4j import criar_driver
from src.consultas import consultar

pd.set_option("display.max_colwidth", 80)
driver = criar_driver()

## 1. Carga do grafo

As tabelas de nós e relações são montadas a partir dos CSVs e enviadas ao Neo4j. Toda escrita usa `MERGE`: rodar a carga de novo não duplica nada.

In [2]:
from src import preparar_grafo
from src.grafo import carregar, contar

preparar_grafo.main()
print()
carregar(driver)
pd.Series(contar(driver), name="total").to_frame()

paises: 10 linhas
empresas: 15 linhas
aparelhos: 19 linhas
leis: 11 linhas
criterios: 12 linhas
lei_criterio: 38 linhas
respostas: 180 linhas
notas: 15 linhas
dados_coletados: 210 linhas
terceiros: 175 linhas
paises_dos_dados: 21 linhas
protecao: 5 linhas



01_paises.cypher: 10 linhas enviadas


02_empresas.cypher: 15 linhas enviadas


03_aparelhos.cypher: 19 linhas enviadas


04_leis.cypher: 11 linhas enviadas


05_criterios.cypher: 12 linhas enviadas


... (4 outputs left out)

10_terceiros.cypher: 175 linhas enviadas


11_paises_dos_dados.cypher: 21 linhas enviadas


12_protecao.cypher: 5 linhas enviadas


,total
(:Aparelho),19
(:Criterio),12
(:Dado),9
(:Empresa),15
(:Lei),11
(:Pais),10
(:Terceiro),8
[:ARMAZENA_EM],8
[:COLETA],210
[:COMPARTILHA_COM],175


## 2. Validação do grafo

Antes de ler qualquer resultado, o grafo é conferido contra as tabelas de origem: a contagem de cada rótulo e relação, e cada consulta das hipóteses recalculada de forma independente com pandas. Se o grafo e o pandas divergirem, a célula abaixo falha.

In [3]:
from src.validar_grafo import ESPERADOS, validar

erros = validar(driver)
assert not erros, erros
print(f"Grafo válido: contagens e {len(ESPERADOS)} consultas conferem com os CSVs.")

Grafo válido: contagens e 6 consultas conferem com os CSVs.


## 3. H1 — dado neural como categoria específica

*A maioria das políticas não trata dado neural como categoria específica.* Critérios C01 (a política nomeia o dado neural) e C02 (dá a ele proteção reforçada).

In [4]:
consultar(driver, "h1_categoria_especifica")

,criterio,nome,grupo,politicas,sim,nao,nao_menciona
0,C01,Categoria específica,brasil,9,6,0,3
1,C01,Categoria específica,comparacao,6,5,0,1
2,C02,Proteção reforçada,brasil,9,2,1,6
3,C02,Proteção reforçada,comparacao,6,4,0,2


**Leitura (execução de 23/11/2026).** A hipótese se divide em duas:

- **Nomear:** 6 das 9 políticas do grupo brasil citam o dado neural com nome próprio (C01). Nesse sentido estrito, H1 **não se sustenta**.
- **Proteger:** só 2 das 9 dão ao dado neural proteção reforçada (C02); as outras 7 o tratam como qualquer dado ou não dizem nada. Nesse sentido, H1 **se sustenta**.

Nomear o dado neural, portanto, não costuma vir acompanhado de proteção diferente.

## 4. H2 — terceiros e países dos dados

*Parte relevante das empresas compartilha dados com terceiros ou guarda os dados fora do Brasil.* Primeiro, os fatos extraídos das políticas; depois, as respostas aos critérios C06 a C08.

Atender a ordem de autoridade (polícia, Justiça) aparece em quase toda política e não é o compartilhamento que H2 mede, por isso há uma coluna que o exclui. Uma política que não cita país não guarda necessariamente os dados no Brasil: ela aparece em `sem_pais_declarado`.

In [5]:
consultar(driver, "h2_terceiros_e_paises")

,grupo,politicas,compartilha_com_terceiros,compartilha_exceto_autoridades,compartilha_com_publicidade,dados_fora_do_brasil,sem_pais_declarado,terceiros_ou_fora
0,brasil,9,9,9,7,6,3,9
1,comparacao,6,5,5,4,4,2,5


In [6]:
consultar(driver, "h2_criterios")

,criterio,nome,grupo,politicas,sim,nao,nao_menciona
0,C06,Sem venda nem publicidade,brasil,9,0,6,3
1,C06,Sem venda nem publicidade,comparacao,6,2,2,2
2,C07,Terceiros identificados,brasil,9,3,0,6
3,C07,Terceiros identificados,comparacao,6,4,0,2
4,C08,Local de armazenamento,brasil,9,2,0,7
5,C08,Local de armazenamento,comparacao,6,0,1,5


**Leitura.** Todas as 9 políticas do grupo brasil preveem compartilhamento com terceiros além das autoridades, e 7 citam parceiros de publicidade. Nenhuma garante não vender nem usar os dados para publicidade (C06: 6 `nao`, 3 em silêncio). 6 das 9 citam países fora do Brasil como destino dos dados, e nenhuma cita o Brasil. H2 **se sustenta**.

## 5. H3 — garantias frente às leis de referência

*As políticas oferecem menos garantias do que as leis de referência exigem.* Primeiro, a proporção de `sim` em cada critério de H3; depois, para cada norma em vigor que protege o sinal neural de forma específica, quantas respostas do grupo brasil ficam abaixo do que ela fundamenta (lacunas).

In [7]:
consultar(driver, "h3_criterios")

,criterio,nome,grupo,politicas,sim,pct_sim
0,C03,Consentimento específico,brasil,9,0,0.0
1,C03,Consentimento específico,comparacao,6,1,16.7
2,C04,Finalidades declaradas,brasil,9,5,55.6
3,C04,Finalidades declaradas,comparacao,6,5,83.3
4,C05,Inferências informadas,brasil,9,4,44.4
5,C05,Inferências informadas,comparacao,6,4,66.7
6,C09,Exclusão a pedido,brasil,9,6,66.7
7,C09,Exclusão a pedido,comparacao,6,2,33.3
8,C10,Prazo de retenção,brasil,9,4,44.4
9,C10,Prazo de retenção,comparacao,6,2,33.3


In [8]:
consultar(driver, "h3_lacunas_por_lei")

,lei,jurisdicao,criterios_fundamentados,respostas,lacunas,pct_lacunas
0,CA-SB-1223,Califórnia,8,72,44,61.1
1,CL-LEY-21383,nacional,2,18,10,55.6
2,CL-ROL-105065-2023,nacional,2,18,10,55.6
3,CO-HB-24-1058,Colorado,8,72,48,66.7


**Leitura.** Nenhuma política do grupo brasil pede consentimento específico para o dado neural (C03) nem garante os direitos a quem mora no Brasil (C12). Frente às leis de Colorado e Califórnia, mais de 60% das respostas ficam abaixo do que elas fundamentam. H3 **se sustenta**, com a ressalva de que as leis estrangeiras não obrigam empresas a atender o Brasil: a comparação mede distância de um padrão, não descumprimento.

## 6. Pergunta complementar — os dois grupos

*As políticas dos aparelhos que chegam ao Brasil oferecem menos garantias do que as dos aparelhos vendidos só em mercados com lei de dados neurais?*

In [9]:
consultar(driver, "complementar_notas")

,grupo,politicas,nota_minima,nota_mediana,nota_media,nota_maxima_obtida,nota_maxima_possivel
0,brasil,9,0,5.0,7.7,16,25
1,comparacao,6,1,11.0,11.3,21,25


**Leitura.** A mediana da nota é 5 no grupo brasil e 11 no grupo de comparação (máximo 25). A diferença vai na direção esperada, mas os grupos são pequenos (9 e 6) e o grupo brasil é o conjunto de todos os casos localizados, não uma amostra. Por isso a comparação é descritiva: não há teste de significância, que daria uma falsa ideia de inferência para uma população.

## 7. Para onde vai o sinal neural

Consulta que só o grafo responde de forma direta: destinos dos dados das empresas do grupo brasil que coletam sinal neural, cruzados com as normas de proteção específica em vigor em cada destino.

In [10]:
consultar(driver, "fluxo_sinal_neural")

,destino,tipo,empresas,ids,relacoes,normas_especificas
0,Estados Unidos,pais,5,P01;P03;P05;P07;P08,ARMAZENA_EM;TRANSFERE_PARA,CA-SB-1223 (Califórnia);CO-HB-24-1058 (Colorado)
1,Canadá,pais,3,P01;P07;P08,ARMAZENA_EM;TRANSFERE_PARA,
2,Espaço Econômico Europeu,regiao,2,P01;P07,ARMAZENA_EM;TRANSFERE_PARA,
3,Austrália,pais,1,P03,TRANSFERE_PARA,
4,União Europeia,regiao,1,P08,TRANSFERE_PARA,
5,Vietnã,pais,1,P03,TRANSFERE_PARA,


**Leitura.** Os Estados Unidos são o destino mais citado (5 empresas). Lá as normas de proteção específica são estaduais (Colorado e Califórnia): estar no país não garante que o dado fique sob uma delas. Nenhum outro destino tem norma específica em vigor no grafo.

## Limitações

- As respostas e os fatos vêm do modelo de linguagem; a validação contra a avaliação manual está pendente (#16).
- A relação `PROTEGE` é derivada da tabela de leis e cobre só o sinal neural; a cobertura das inferências depende da revisão jurídica (#6).
- Os nós `:Terceiro` são categorias: o nome citado por cada política fica na relação, e a categoria `outros` reúne casos diversos.
- Os grupos são pequenos; os resultados descrevem estes casos e não se generalizam.

In [11]:
driver.close()